In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import pandas as pd
from Constants import Const
import SymptomPreprocessing as Symptom
import re

In [3]:
new_dvh_folder_path = "../DVH_20260421/"
new_mdasi_path = "../PA140947PROFHutcheso-CDFUICRequest_DATA_2025-11-18_0858 UIC_PIVOT+DATES_to_DAYS_INCLUDING_STIEFEL_IDs_Anonymized.xlsx"

In [4]:
folder = Path(new_dvh_folder_path)
csv_files = sorted(folder.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {folder}")


def normalize_id_series(series):
    return series.dropna().astype(str).str.strip()


def load_reference_ids():
    mdasi_path = Path(new_mdasi_path)
    if mdasi_path.exists():
        try:
            reference_df = pd.read_excel(mdasi_path)
            if "STIEFEL_ID" in reference_df.columns:
                return (
                    set(normalize_id_series(reference_df["STIEFEL_ID"])),
                    "new_mdasi_path.STIEFEL_ID",
                )
        except Exception:
            pass

    csv_reference_ids = set()
    for csv_path in csv_files:
        reference_df = pd.read_csv(csv_path)
        if "STIEFEL_ID" in reference_df.columns:
            csv_reference_ids.update(normalize_id_series(reference_df["STIEFEL_ID"]))
    if csv_reference_ids:
        return csv_reference_ids, "CSV.STIEFEL_ID"

    return set(), None


reference_ids, reference_source = load_reference_ids()
if not reference_ids:
    raise ValueError("No STIEFEL_ID values were found in the reference data.")

results = []
for csv_path in csv_files:
    df = pd.read_csv(csv_path)
    id_columns = [
        column_name
        for column_name in df.columns
        if column_name.lower() == "id" or column_name.lower().endswith("_id")
    ]

    for id_column in id_columns:
        values = normalize_id_series(df[id_column])
        unique_values = set(values)
        missing_values = sorted(unique_values - reference_ids)

        results.append(
            {
                "file": csv_path.name,
                "id_column": id_column,
                "unique_id_count": len(unique_values),
                "missing_count": len(missing_values),
                "missing_sample": missing_values[:20],
            }
        )

summary = pd.DataFrame(results)

print(f"Reference source: {reference_source}")

if summary.empty:
    print("No ID columns were found.")
else:
    display(summary)

    files_with_missing = summary.loc[summary["missing_count"] > 0]
    if files_with_missing.empty:
        print("All ID values are present in the reference STIEFEL_ID set.")
    else:
        print("Some ID values are missing from the reference STIEFEL_ID set.")
        for _, row in files_with_missing.iterrows():
            print(
                f"{row['file']} | {row['id_column']} | missing={row['missing_count']} | sample={row['missing_sample']}"
            )

Reference source: new_mdasi_path.STIEFEL_ID


,file,id_column,unique_id_count,missing_count,missing_sample
0,Folder_1_DVH_all.csv,id,50,0,[]
1,Folder_2_DVH_all.csv,id,50,0,[]
2,Folder_3_DVH_all.csv,id,50,0,[]
3,Folder_4_DVH_all.csv,id,50,0,[]
4,Folder_5_DVH_all.csv,id,50,0,[]
5,Folder_6_DVH_all.csv,id,50,0,[]
6,Folder_7_DVH_all.csv,id,19,0,[]


All ID values are present in the reference STIEFEL_ID set.


In [5]:
organ_rename_dict = {
    "cricoid": "Cricoid_cartilage",
    "cricopharyngeus": "Cricopharyngeal_Muscle",
    "esophagus_u": "Esophagus",
    "oral_cavity": "Extended_Oral_Cavity",
    "musc_geniogloss": "Genioglossus_M",
    "hardpalate": "Hard_Palate",
    "bone_hyoid": "Hyoid_bone",
    "musc_constrict_i": "IPC",
    "lips_lower": "Lower_Lip",
    "lips_upper": "Upper_Lip",
    "musc_constrict_m": "MPC",
    "musc_mgh_complex": "Mylogeniohyoid_M",
    "musc_mghcomplex": "Mylogeniohyoid_M",
    "palate_soft": "Soft_Palate",
    "musc_constrict_s": "SPC",
    "spinalcord_cerv": "Spinal_Cord",
    "larynx_sg": "Supraglottic_Larynx",
    "cartlg_thyroid": "Thyroid_cartilage",
    "brachial_plex_r": "Rt_Brachial_Plexus",
    "brachial_plex_l": "Lt_Brachial_Plexus",
    "pterygoid_lat_r": "Rt_Lateral_Pterygoid_M",
    "pterygoid_lat_l": "Lt_Lateral_Pterygoid_M",
    "musc_masseter_r": "Rt_Masseter_M",
    "musc_masseter_l": "Lt_Masseter_M",
    "bone_mastoid_r": "Rt_Mastoid",
    "bone_mastoid_l": "Lt_Mastoid",
    "pterygoid_med_r": "Rt_Medial_Pterygoid_M",
    "pterygoid_med_l": "Lt_Medial_Pterygoid_M",
    "parotid_r": "Rt_Parotid_Gland",
    "parotid_l": "Lt_Parotid_Gland",
    "musc_sclmast_r": "Rt_Sternocleidomastoid_M",
    "musc_sclmast_l": "Lt_Sternocleidomastoid_M",
    "glnd_submand_r": "Rt_Submandibular_Gland",
    "glnd_submand_l": "Lt_Submandibular_Gland",
    "musc_digastric_ra": "Rt_Ant_Digastric_M",
    "musc_digastric_la": "Lt_Ant_Digastric_M",
    "musc_digastric_rp": "Rt_Post_Digastric_M",
    "musc_digastric_lp": "Lt_Post_Digastric_M",
    "Esophagus_U": "Esophagus",
    "esophagus": "Esophagus",
    "Esophagus_up": "Esophagus",
    "Esophagus_Up": "Esophagus",
    "cool down eso": "Esophagus",
    "fs shaper esoph": "Esophagus",
    "Partial Esophagus": "Esophagus",
    "fsPushEso": "Esophagus",
    "z_eso push": "Esophagus",
    "fs shaper eso": "Esophagus",
    "fsEsophMax28": "Esophagus",
    "fsEsophMax30": "Esophagus",
    "FS esoph opt": "Esophagus",
    "eso sub": "Esophagus",
    "fs esoph hot": "Esophagus",
    "lower esophagus": "Esophagus",
    "SpinalCord": "Spinal_Cord",
    "SpinalCord_PRV05": "Spinal_Cord",
    "SpinalCord_Cerv": "Spinal_Cord",
    "SpinalCord_PRV5": "Spinal_Cord",
    "SpinalCordPRV_05": "Spinal_Cord",
    "SpinalCord_05": "Spinal_Cord",
    "SpinalCord_PRV02": "Spinal_Cord",
    "SpinalCord_03": "Spinal_Cord",
    "BrachialPlex_L": "Lt_Brachial_Plexus",
    "fsL_BrachPlex exp": "Lt_Brachial_Plexus",
    "BrachialPlex_left": "Lt_Brachial_Plexus",
    "Lt Plexus hot": "Lt_Brachial_Plexus",
    "Brachial_Plex_L": "Lt_Brachial_Plexus",
    "Brach Plex_L": "Lt_Brachial_Plexus",
    "Lt brachial plexus 58": "Lt_Brachial_Plexus",
    "Lt BrachialPlex slice 81 and below": "Lt_Brachial_Plexus",
    "BrachialPlex_L_lower": "Lt_Brachial_Plexus",
    "BrachialPlex_L Inferior": "Lt_Brachial_Plexus",
    "Brachial_Plex_R": "Rt_Brachial_Plexus",
    "BrachialPlex_R": "Rt_Brachial_Plexus",
    "BrachialPlex_right": "Rt_Brachial_Plexus",
    "Brachial_R Expanded 2mm": "Rt_Brachial_Plexus",
    "Rplexusexp1mm": "Rt_Brachial_Plexus",
    "Rt plexus hot": "Rt_Brachial_Plexus",
    "Brach Plex_R": "Rt_Brachial_Plexus",
    "rt brachial plexus 58": "Rt_Brachial_Plexus",
    "Rt_BrachialPlex slice 81 and below": "Rt_Brachial_Plexus",
    "BrachialPlex R": "Rt_Brachial_Plexus",
    "BrachialPlex_R lower": "Rt_Brachial_Plexus",
    "cricopharyngeus": "Cricopharyngeal_Muscle",
    "Cricopharyngeus": "Cricopharyngeal_Muscle",
    "Cricopharyngeus & Arytenoids": "Cricopharyngeal_Muscle",
    "Cricopharyngeus & Arytenoid": "Cricopharyngeal_Muscle",
    "Arytenoid & Cricopharyngeus": "Cricopharyngeal_Muscle",
    "Cricoid": "Cricoid_cartilage",
    "Musc_Constrict_I": "IPC",
    "Musc_Constrict_M": "MPC",
    "Musc_Constrict_S": "SPC",
    "Brainstem": "Brainstem",
    "Brainstem_PRV05": "Brainstem",
    "BrainStem": "Brainstem",
    "BrainStem_PRV5": "Brainstem",
    "Brainstem_PRV5": "Brainstem",
    "BrainStem_05": "Brainstem",
    "Brainstem_03": "Brainstem",
    "Brainstem Expanded 5mm": "Brainstem",
    "Brainstem expanded 5mm": "Brainstem",
    "BrainStem_03": "Brainstem",
    "cool down brainstem": "Brainstem",
    "cool down cord_brainstem": "Brainstem",
    "Brainstem_PRV03": "Brainstem",
    "Brainstem_PRV02": "Brainstem",
    "Brainstem +0.5": "Brainstem",
    "Larynx": "Larynx",
    "larynx sub": "Larynx",
    "Larynx_SG": "Larynx",
    "fslarynxhot": "Larynx",
    "Larynx Proper": "Larynx",
    "Larynx proper": "Larynx",
    "FS Larynx Opt": "Larynx",
    "fsLarynxStrip2": "Larynx",
    "fslarynxexp": "Larynx",
    "fsLarynx_plan": "Larynx",
    "larynx": "Larynx",
    "fsCool larynx": "Larynx",
    "Larynx sub": "Larynx",
    "post larynx": "Larynx",
    "pLarynx": "Larynx",
    "dgLarynx_42": "Larynx",
    "xLarynx": "Larynx",
    "z_larynx push": "Larynx",
    "fs_larynx hard": "Larynx",
    "fs larynx av": "Larynx",
    "fs Larynx av": "Larynx",
    "fs shaper larynx": "Larynx",
    "fs larynx shaper": "Larynx",
    "fs larynx shaper2": "Larynx",
    "fs ant larynx": "Larynx",
    "z_hardlarynx": "Larynx",
    "fs larynx horn": "Larynx",
    "FS larynx opt": "Larynx",
    "fs larynx push": "Larynx",
    "fs Avoivd Eso Larynx and Thryoid": "Larynx",
    "FS Larynx opt": "Larynx",
    "fs larynx original": "Larynx",
    "fs no30_larynx": "Larynx",
    "Cartlg_Thyroid": "Thyroid_cartilage",
    "Glnd_Thyroid_L": "Thyroid_cartilage",
    "Glnd_Thyroid_R": "Thyroid_cartilage",
    "Thyroid": "Thyroid_cartilage",
    "thyroid": "Thyroid_cartilage",
    "fs thyroid push": "Thyroid_cartilage",
    "Glnd_Thyroid": "Thyroid_cartilage",
    "fs_thyroid push": "Thyroid_cartilage",
    "thyroid sub": "Thyroid_cartilage",
    "Thryoid": "Thyroid_cartilage",
    "Musc_Sclmast_R": "Rt_Sternocleidomastoid_M",
    "Bone_Mastoid_R": "Rt_Mastoid",
    "fs mastoid push": "Rt_Mastoid",
    "fs r mastoid push": "Rt_Mastoid",
    "Parotid_R": "Rt_Parotid_Gland",
    "fsParotid_R_Sub": "Rt_Parotid_Gland",
    "fs rt parotid push": "Rt_Parotid_Gland",
    "fsRparotidhot": "Rt_Parotid_Gland",
    "fsParotid_R_Sub_1": "Rt_Parotid_Gland",
    "fsRparotid_out": "Rt_Parotid_Gland",
    "fsCool rt parotid": "Rt_Parotid_Gland",
    "fs rt parotid low push": "Rt_Parotid_Gland",
    "R Parotid push": "Rt_Parotid_Gland",
    "Rt Parotid top": "Rt_Parotid_Gland",
    "fsRt Parotid push": "Rt_Parotid_Gland",
    "fsRtParotid10Gypush": "Rt_Parotid_Gland",
    "Rt Parotid push": "Rt_Parotid_Gland",
    "fs rt parotid sub": "Rt_Parotid_Gland",
    "fs parotid_Rsub": "Rt_Parotid_Gland",
    "Sub_Parotid_R": "Rt_Parotid_Gland",
    "rt parotid push": "Rt_Parotid_Gland",
    "z_parotidR_in70": "Rt_Parotid_Gland",
    "z_parotid_R_push": "Rt_Parotid_Gland",
    "R parotid push": "Rt_Parotid_Gland",
    "fs_rparotid top": "Rt_Parotid_Gland",
    "fs parotids push": "Rt_Parotid_Gland",
    "Parotid_Critical_R": "Rt_Parotid_Gland",
    "fs Parotid Top": "Rt_Parotid_Gland",
    "Pterygoid_Med_R": "Rt_Medial_Pterygoid_M",
    "Pterygoid_Lat_R": "Rt_Lateral_Pterygoid_M",
    "Musc_Masseter_R": "Rt_Masseter_M",
    "Musc_Sclmast_L": "Lt_Sternocleidomastoid_M",
    "Bone_Mastoid_L": "Lt_Mastoid",
    "Parotid_L": "Lt_Parotid_Gland",
    "fsParotid_L_Sub": "Lt_Parotid_Gland",
    "fs lt parotid push": "Lt_Parotid_Gland",
    "fsLparotidhot": "Lt_Parotid_Gland",
    "Lt parotid push2": "Lt_Parotid_Gland",
    "fs Lt parotid push": "Lt_Parotid_Gland",
    "fsParotid_L_Sub_1": "Lt_Parotid_Gland",
    "fs lt parotid low push": "Lt_Parotid_Gland",
    "L Parotid push": "Lt_Parotid_Gland",
    "Lt Parotid top": "Lt_Parotid_Gland",
    "fsParotid_L_Sub1": "Lt_Parotid_Gland",
    "fsParotid_L_Sub2": "Lt_Parotid_Gland",
    "fs Lt Parotid Push": "Lt_Parotid_Gland",
    "fa lt parotid push": "Lt_Parotid_Gland",
    "fs lt parotid": "Lt_Parotid_Gland",
    "lt parotid push": "Lt_Parotid_Gland",
    "fs lt parotid sub": "Lt_Parotid_Gland",
    "fs lt parotid push 2": "Lt_Parotid_Gland",
    "fs parotid_Lsub": "Lt_Parotid_Gland",
    "Sub_Parotid_L": "Lt_Parotid_Gland",
    "L parotid sub": "Lt_Parotid_Gland",
    "Lt Parotid push": "Lt_Parotid_Gland",
    "Parotid_Critical_L": "Lt_Parotid_Gland",
    "New Lt Parotid Sub": "Lt_Parotid_Gland",
    "Glnd_Submand_L": "Lt_Submandibular_Gland",
    "Submandibular_L": "Lt_Submandibular_Gland",
    "left submandibular gland": "Lt_Submandibular_Gland",
    "Submandibular Gland_L": "Lt_Submandibular_Gland",
    "Submandubular_L": "Lt_Submandibular_Gland",
    "l submandibular": "Lt_Submandibular_Gland",
    "z_submand L_push": "Lt_Submandibular_Gland",
    "DNU Glnd_Submand_L": "Lt_Submandibular_Gland",
    "glnd_submand_l": "Lt_Submandibular_Gland",
    "Pterygoid_Med_L": "Lt_Medial_Pterygoid_M",
    "Pterygoid_Lat_L": "Lt_Lateral_Pterygoid_M",
    "Musc_Masseter_L": "Lt_Masseter_M",
    "larynx_sg": "Supraglottic_Larynx",
    "glnd_submand_r": "Rt_Submandibular_Gland",
    "Glnd_Submand_R": "Rt_Submandibular_Gland",
    "Submandibular_R": "Rt_Submandibular_Gland",
    "z_submand push_R": "Rt_Submandibular_Gland",
    "DNU Glnd_Submand_R": "Rt_Submandibular_Gland",
    "Bone_Hyoid": "Hyoid_bone",
    "Palate_Soft": "Soft_Palate",
    "5700 (fff Soft Palate Aprvd LLM)": "Soft_Palate",
    "6300 (fff Soft Palate Aprvd LLM)": "Soft_Palate",
    "7000 (fff Soft Palate Aprvd LLM)": "Soft_Palate",
    "Musc_Geniogloss": "Genioglossus_M",
    "Tongue": "Tongue",
    "tongue": "Tongue",
    "4500 (fff Base of Tongue JPR)": "Tongue",
    "5700 (fff Base of Tongue JPR)": "Tongue",
    "6300 (fff Base of Tongue JPR)": "Tongue",
    "6600 (fff Base of Tongue JPR)": "Tongue",
    "7000 (fff Base of Tongue JPR)": "Tongue",
    "fs tongue push": "Tongue",
    "Musc_Digastric_RA": "Rt_Ant_Digastric_M",
    "Musc_Digastric_RP": "Rt_Ant_Digastric_M",
    "Musc_Digastric_LA": "Lt_Ant_Digastric_M",
    "Musc_Digastric_LP": "Lt_Ant_Digastric_M",
    "Musc_MGHComplex": "Mylogeniohyoid_M",
    "Cavity_Oral": "Extended_Oral_Cavity",
    "Oral_Cavity": "Extended_Oral_Cavity",
    "OralCavity": "Extended_Oral_Cavity",
    "CavityOral": "Extended_Oral_Cavity",
    "Oral Cavity": "Extended_Oral_Cavity",
    "OralCavity_Original": "Extended_Oral_Cavity",
    "fsOralCavity": "Extended_Oral_Cavity",
    "oral cavity": "Extended_Oral_Cavity",
    "zCavity_Oral (1)": "Extended_Oral_Cavity",
    "xCavity_Oral": "Extended_Oral_Cavity",
    "Cavity_Oral push": "Extended_Oral_Cavity",
    "z_Cavity_Oral_push": "Extended_Oral_Cavity",
    "Bone_Mandible": "Mandible",
    "Mandible": "Mandible",
    "fsCool mandible": "Mandible",
    "fs7300mandible": "Mandible",
    "z_mandible IN": "Mandible",
    "fs_73mandible": "Mandible",
    "fs_hotmandible": "Mandible",
    "fsMandiblePush": "Mandible",
    "fs mandible hot": "Mandible",
    "fs mandible": "Mandible",
    "zMaxmandible": "Mandible",
    "mandible avd": "Mandible",
    "fsMandible sub": "Mandible",
    "Hardpalate": "Hard_Palate",
    "Lips_Lower": "Lower_Lip",
    "Lips_Upper": "Upper_Lip",
}

In [6]:
folder = Path(new_dvh_folder_path)
csv_files = sorted(folder.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {folder}")

raw_frames = [
    pd.read_csv(csv_path).assign(source_file=csv_path.name) for csv_path in csv_files
]
merged_dvh_df = pd.concat(raw_frames, ignore_index=True)

required_columns = {"id", "Structure"}
missing_required_columns = required_columns - set(merged_dvh_df.columns)
if missing_required_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_required_columns)}")

merged_dvh_df["id"] = merged_dvh_df["id"].astype(str).str.strip()
merged_dvh_df["Structure"] = merged_dvh_df["Structure"].astype(str).str.strip()
merged_dvh_df["organ"] = merged_dvh_df["Structure"].replace(organ_rename_dict)
merged_dvh_df = merged_dvh_df.drop_duplicates(subset=["id", "organ"], keep="first").copy()

unmapped_structures = sorted(
    set(
        merged_dvh_df.loc[
            merged_dvh_df["organ"] == merged_dvh_df["Structure"], "Structure"
        ]
    )
    - set(organ_rename_dict.keys())
)

expected_organs = list(Const.organ_list)
patient_organs = merged_dvh_df.groupby("id")["organ"].apply(
    lambda values: sorted(set(values.dropna()))
)

patient_summary_rows = []
for patient_id, organs in patient_organs.items():
    organ_set = set(organs)
    missing_organs = [organ for organ in expected_organs if organ not in organ_set]
    extra_organs = sorted(organ_set - set(expected_organs))
    patient_summary_rows.append(
        {
            "id": patient_id,
            "present_organ_count": len(organ_set),
            "missing_organ_count": len(missing_organs),
            "missing_organs": missing_organs,
            "extra_organs": extra_organs,
        }
    )

patient_summary = pd.DataFrame(patient_summary_rows).sort_values(
    ["missing_organ_count", "id"], ascending=[False, True]
)

print(f"Merged rows after deduplication: {len(merged_dvh_df)}")
print(
    f"Patients found: {patient_summary['id'].nunique() if not patient_summary.empty else 0}"
)

# if unmapped_structures:
#     print("Unmapped structure names found:")
#     for structure_name in unmapped_structures:
#         print(f"  {structure_name}")
# else:
#     print(
#         "All structure names were mapped by organ_rename_dict or were already normalized."
#     )

if patient_summary.empty:
    print("No patient records were found after merging.")
else:
    patients_with_missing = patient_summary.loc[
        patient_summary["missing_organ_count"] > 0
    ]
    print(f"Patients missing organs: {len(patients_with_missing)}")

    if patients_with_missing.empty:
        print("Every patient has all organs in Const.organ_list.")
    else:
        print("Patients missing organs:")
        for _, row in patients_with_missing.iterrows():
            print(
                f"id={row['id']} | missing={row['missing_organ_count']} | missing_organs={row['missing_organs']}"
            )

Merged rows after deduplication: 36096
Patients found: 319
Patients missing organs: 319
Patients missing organs:
id=STIEFEL_792 | missing=9 | missing_organs=['Rt_Mastoid', 'Rt_Lateral_Pterygoid_M', 'Lt_Mastoid', 'Lt_Medial_Pterygoid_M', 'Lt_Lateral_Pterygoid_M', 'Supraglottic_Larynx', 'Soft_Palate', 'Hard_Palate', 'Upper_Lip']
id=STIEFEL_1010 | missing=1 | missing_organs=['Supraglottic_Larynx']
id=STIEFEL_1015 | missing=1 | missing_organs=['Supraglottic_Larynx']
id=STIEFEL_1022 | missing=1 | missing_organs=['Supraglottic_Larynx']
id=STIEFEL_1023 | missing=1 | missing_organs=['Supraglottic_Larynx']
id=STIEFEL_1028 | missing=1 | missing_organs=['Supraglottic_Larynx']
id=STIEFEL_1030 | missing=1 | missing_organs=['Supraglottic_Larynx']
id=STIEFEL_1031 | missing=1 | missing_organs=['Supraglottic_Larynx']
id=STIEFEL_1033 | missing=1 | missing_organs=['Supraglottic_Larynx']
id=STIEFEL_1034 | missing=1 | missing_organs=['Supraglottic_Larynx']
id=STIEFEL_1035 | missing=1 | missing_organs=['Sup

In [7]:
# Re-merge DVH CSVs, rename Structure values using organ_rename_dict,
# and keep only rows whose Structure is in Const.organ_list. Columns unchanged.
folder = Path(new_dvh_folder_path)
csv_files = sorted(folder.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {folder}")
df_list = [pd.read_csv(p) for p in csv_files]
merged_dvh_renamed = pd.concat(df_list, ignore_index=True)
required_columns = {"id", "Structure"}
missing_required_columns = required_columns - set(merged_dvh_renamed.columns)
if missing_required_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_required_columns)}")
merged_dvh_renamed["id"] = merged_dvh_renamed["id"].astype(str).str.strip()
merged_dvh_renamed["Structure"] = merged_dvh_renamed["Structure"].astype(str).str.strip()
# Replace Structure values using provided mapping; this updates the Structure column in-place
merged_dvh_renamed["Structure"] = merged_dvh_renamed["Structure"].replace(organ_rename_dict)
# Keep only rows where the (renamed) Structure is one of the expected organs
filtered = merged_dvh_renamed[merged_dvh_renamed["Structure"].isin(Const.organ_list)].copy()
print(f"Total rows before filter: {len(merged_dvh_renamed)}")
print(f"Total rows after filter (before dedup): {len(filtered)}")
filtered = filtered.drop_duplicates(subset=["id", "Structure"], keep="first").copy()
print(f"Total rows after deduplication (id, Structure): {len(filtered)}")
print(f"Unique patients after filter: {filtered['id'].nunique()}")
filtered

Total rows before filter: 40747
Total rows after filter (before dedup): 17084
Total rows after deduplication (id, Structure): 12433
Unique patients after filter: 319


,id,Structure,Volume,mean,minGy,maxGy,V5,V10,V15,V20,...,D65,D70,D75,D80,D85,D90,D95,D97,D98,D99
3,STIEFEL_1034,Hyoid_bone,3.5201,70.9873,69.9,71.7,100.0000,100.0000,100.0000,100.0000,...,70.9089,70.8640,70.8133,70.7627,70.7121,70.6237,70.5234,70.4502,70.3906,70.3310
4,STIEFEL_1034,Mandible,98.2159,37.1050,3.5,70.9,99.7285,96.0087,93.7408,89.9127,...,27.2509,25.3946,23.8739,22.8103,21.7711,19.9465,11.8452,8.5224,7.1977,5.9667
6,STIEFEL_1034,Lt_Mastoid,1.7854,41.1013,26.1,50.7,100.0000,100.0000,100.0000,100.0000,...,39.6788,38.7900,37.4250,36.1320,34.5117,32.4300,30.1900,28.8935,28.2580,27.4290
7,STIEFEL_1034,Rt_Mastoid,0.9653,36.5273,23.5,46.3,100.0000,100.0000,100.0000,100.0000,...,34.8400,34.1900,33.2250,32.3800,31.0950,29.3400,27.5300,26.4580,25.4720,24.0720
8,STIEFEL_1034,Lt_Brachial_Plexus,6.0615,47.2778,6.7,62.9,100.0000,98.9421,95.1281,91.2027,...,48.6482,47.3640,46.0385,39.3400,29.2400,21.4800,15.1533,12.5940,11.4280,9.8920
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40708,STIEFEL_2397,Rt_Lateral_Pterygoid_M,10.0946,18.6586,5.5,32.5,100.0000,95.8877,70.4280,44.0154,...,16.0783,15.0776,14.1625,13.3985,12.6156,11.7881,10.4506,9.3780,8.7637,7.4371
40709,STIEFEL_2397,Lt_Medial_Pterygoid_M,12.4403,53.1266,24.7,72.1,100.0000,100.0000,100.0000,100.0000,...,50.2683,47.9257,45.0700,42.2022,39.0267,34.5900,29.5660,27.8395,27.0620,25.9744
40710,STIEFEL_2397,Rt_Medial_Pterygoid_M,12.2647,36.8865,23.1,53.5,100.0000,100.0000,100.0000,100.0000,...,34.0661,33.1240,32.2051,31.1846,30.0300,28.7650,27.1970,26.4296,25.8335,25.1669
40712,STIEFEL_2397,Spinal_Cord,16.3856,10.4861,0.1,23.1,81.9464,48.4346,19.4233,5.4377,...,9.2766,9.1087,8.9173,8.1486,3.0891,1.9821,1.4264,1.2528,1.1344,0.7431


In [8]:
filtered["id"] = (
    filtered["id"]
    .astype(str) 
    .str.replace("STIEFEL_", "", regex=False)
    .astype(int)
)

filtered["DicomType"] = "ORGAN"
# filtered
filtered.to_csv("../data/dvh_20260421.csv", index=False)

In [9]:
mdasi_df = pd.read_excel('../PA140947PROFHutcheso-CDFUICRequest_DATA_2025-11-18_0858 UIC_PIVOT+DATES_to_DAYS_INCLUDING_STIEFEL_IDs_Anonymized.xlsx')
mdasi_df

,#,STIEFEL_ID,consent_age,gender,ethnicity,race,smoking_status,tobacco_packs_per_day,tobacco_used_years,pack_years,...,2_wks_after_primar_arm_7__eq5d_complete,3_wks_after_primar_arm_7__eq5d_complete,4_wks_after_primar_arm_7__eq5d_complete,5_wks_after_primar_arm_7__eq5d_complete,6_months_fitbit_arm_7__eq5d_complete,day_1_chemo_arm_8__eq5d_complete,day_29_chemo_arm_8__eq5d_complete,presx_arm_8__eq5d_complete,12_month_fitbit_arm_7__eq5d_complete,1824_month_fitbit_arm_7__eq5d_complete
0,1,STIEFEL_2412,66.0,M,Not Hispanic or Latino,White,4.0,1.0,4.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,STIEFEL_1728,67.0,F,Not Hispanic or Latino,White,4.0,0.5,5.0,2.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,STIEFEL_1729,81.0,M,Not Hispanic or Latino,White,2.0,1.0,64.4,64.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,STIEFEL_1732,70.0,M,Not Hispanic or Latino,White,5.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,STIEFEL_1734,64.0,F,Not Hispanic or Latino,White,4.0,0.3,4.0,1.2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2878,2879,STIEFEL_2869,64.0,M,Not Hispanic or Latino,White,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2879,2880,STIEFEL_2870,86.0,F,Not Hispanic or Latino,White,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2880,2881,STIEFEL_1764,62.0,M,Not Hispanic or Latino,White,4.0,1.0,50.0,50.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2881,2882,STIEFEL_1444,61.0,M,Not Hispanic or Latino,White,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
timepoint_prefix_map = {
    "baseline_arm_1__mdasi_": "baseline_mdasi_",
    "start_of_xrt_arm_3__mdasi_": "startRT_mdasi_",
    "week_2_arm_3__mdasi_": "wk_2_mdasi_",
    "week_3_arm_3__mdasi_": "wk_3_mdasi_",
    "week_4_arm_3__mdasi_": "wk_4_mdasi_",
    "week_5_arm_3__mdasi_": "wk_5_mdasi_",
    "week_6_arm_3__mdasi_": "wk_6_mdasi_",
    "end_of_xrt_arm_3__mdasi_": "endRT_mdasi_",
    "6_wks_after_primar_arm_6__mdasi_": "wk6_post_mdasi_",
    "12_months_arm_6__mdasi_": "M12_mdasi_",
    "3to6_months_arm_6__mdasi_": "M6_mdasi_",
    "18to24_months_arm_6__mdasi_": "M18_mdasi_",
    "60_months_arm_6__mdasi_": "M60_mdasi_",
}

timepoint_general_activity_prefix_map = {
    "baseline_arm_1__general_activity": "baseline_general_activity",
    "start_of_xrt_arm_3__general_activity": "startRT_general_activity",
    "week_2_arm_3__general_activity": "wk_2_general_activity",
    "week_3_arm_3__general_activity": "wk_3_general_activity",
    "week_4_arm_3__general_activity": "wk_4_general_activity",
    "week_5_arm_3__general_activity": "wk_5_general_activity",
    "week_6_arm_3__general_activity": "wk_6_general_activity",
    "end_of_xrt_arm_3__general_activity": "endRT_general_activity",
    "6_wks_after_primar_arm_6__general_activity": "wk6_post_general_activity",
    "12_months_arm_6__general_activity": "M12_general_activity",
    "3to6_months_arm_6__general_activity": "M6_general_activity",
    "18to24_months_arm_6__general_activity": "M18_general_activity",
    "60_months_arm_6__general_activity": "M60_general_activity",
}


def strip_parenthetical_suffix(column_name):
    return re.sub(r"\s*\(.*\)\s*$", "", str(column_name)).strip()


normalized_column_lookup = {
    strip_parenthetical_suffix(column_name): column_name for column_name in mdasi_df.columns
}


missing_columns = []
for raw_prefix, alias_prefix in timepoint_prefix_map.items():
    for symptom_name in Const.symptoms:
        if symptom_name == "activity":
            continue
        normalized_name = f"{raw_prefix}{symptom_name}"
        if normalized_name not in normalized_column_lookup:
            missing_columns.append(normalized_name)

for raw_name in timepoint_general_activity_prefix_map:
    if raw_name not in normalized_column_lookup:
        missing_columns.append(raw_name)

mdasi_symptom_df = mdasi_df.copy()
mdasi_symptom_df = mdasi_symptom_df.rename(
    columns={
        raw_name: f"{alias_prefix}{symptom_name}"
        for raw_prefix, alias_prefix in timepoint_prefix_map.items()
        for symptom_name in Const.symptoms
        if symptom_name != "activity"
        if (raw_name := normalized_column_lookup.get(f"{raw_prefix}{symptom_name}"))
    }
    | {
        raw_name: alias_prefix
        for raw_name, alias_prefix in timepoint_general_activity_prefix_map.items()
        if raw_name in normalized_column_lookup
    }
)

kept_mdasi_columns = {
    f"{alias_prefix}{symptom_name}"
    for alias_prefix in timepoint_prefix_map.values()
    for symptom_name in Const.symptoms
    if symptom_name != "activity"
}
extra_mdasi_columns = [
    column_name
    for column_name in mdasi_symptom_df.columns
    if "mdasi" in str(column_name).lower() and column_name not in kept_mdasi_columns
]
mdasi_symptom_df = mdasi_symptom_df.drop(columns=extra_mdasi_columns)

# After renaming arm_*/arm_*/__general_activity → short names, remove every other column
# whose name still contains 'activity' (answered flags, eq5d_usualactivity, other trials, …).
kept_general_activity_aliases = set(timepoint_general_activity_prefix_map.values())
extra_activity_columns = [
    column_name
    for column_name in mdasi_symptom_df.columns
    if "activity" in str(column_name).lower()
    and column_name not in kept_general_activity_aliases
]
mdasi_symptom_df = mdasi_symptom_df.drop(columns=extra_activity_columns)

print(f"Total columns: {mdasi_symptom_df.shape[1]}")
print(f"Missing expected columns: {len(missing_columns)}")
print(f"Dropped other mdasi columns: {len(extra_mdasi_columns)}")
print(f"Dropped other 'activity' columns: {len(extra_activity_columns)}")
if missing_columns:
    print("Missing sample:")
    for column_name in missing_columns[:20]:
        print(f"  {column_name}")

mdasi_symptom_df


Total columns: 4507
Missing expected columns: 0
Dropped other mdasi columns: 1539
Dropped other 'activity' columns: 113


,#,STIEFEL_ID,consent_age,gender,ethnicity,race,smoking_status,tobacco_packs_per_day,tobacco_used_years,pack_years,...,2_wks_after_primar_arm_7__eq5d_complete,3_wks_after_primar_arm_7__eq5d_complete,4_wks_after_primar_arm_7__eq5d_complete,5_wks_after_primar_arm_7__eq5d_complete,6_months_fitbit_arm_7__eq5d_complete,day_1_chemo_arm_8__eq5d_complete,day_29_chemo_arm_8__eq5d_complete,presx_arm_8__eq5d_complete,12_month_fitbit_arm_7__eq5d_complete,1824_month_fitbit_arm_7__eq5d_complete
0,1,STIEFEL_2412,66.0,M,Not Hispanic or Latino,White,4.0,1.0,4.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,STIEFEL_1728,67.0,F,Not Hispanic or Latino,White,4.0,0.5,5.0,2.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,STIEFEL_1729,81.0,M,Not Hispanic or Latino,White,2.0,1.0,64.4,64.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,STIEFEL_1732,70.0,M,Not Hispanic or Latino,White,5.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,STIEFEL_1734,64.0,F,Not Hispanic or Latino,White,4.0,0.3,4.0,1.2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2878,2879,STIEFEL_2869,64.0,M,Not Hispanic or Latino,White,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2879,2880,STIEFEL_2870,86.0,F,Not Hispanic or Latino,White,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2880,2881,STIEFEL_1764,62.0,M,Not Hispanic or Latino,White,4.0,1.0,50.0,50.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2881,2882,STIEFEL_1444,61.0,M,Not Hispanic or Latino,White,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
normal_rename_dict = {
    "baseline_arm_1__m": "m_stage",
    "baseline_arm_1__mdadi_global": "baseline_mbs_digest",
    "3to6_months_arm_6__mdadi_global": "M6_mbs_digest",
    "consent_age": "age",
    "gender": "sex",
    "baseline_arm_1__current_status": "concurrent"
}

mdasi_symptom_df = mdasi_symptom_df.rename(columns=normal_rename_dict, inplace=False)
mdasi_symptom_df

,#,STIEFEL_ID,age,sex,ethnicity,race,smoking_status,tobacco_packs_per_day,tobacco_used_years,pack_years,...,2_wks_after_primar_arm_7__eq5d_complete,3_wks_after_primar_arm_7__eq5d_complete,4_wks_after_primar_arm_7__eq5d_complete,5_wks_after_primar_arm_7__eq5d_complete,6_months_fitbit_arm_7__eq5d_complete,day_1_chemo_arm_8__eq5d_complete,day_29_chemo_arm_8__eq5d_complete,presx_arm_8__eq5d_complete,12_month_fitbit_arm_7__eq5d_complete,1824_month_fitbit_arm_7__eq5d_complete
0,1,STIEFEL_2412,66.0,M,Not Hispanic or Latino,White,4.0,1.0,4.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,STIEFEL_1728,67.0,F,Not Hispanic or Latino,White,4.0,0.5,5.0,2.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,STIEFEL_1729,81.0,M,Not Hispanic or Latino,White,2.0,1.0,64.4,64.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,STIEFEL_1732,70.0,M,Not Hispanic or Latino,White,5.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,STIEFEL_1734,64.0,F,Not Hispanic or Latino,White,4.0,0.3,4.0,1.2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2878,2879,STIEFEL_2869,64.0,M,Not Hispanic or Latino,White,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2879,2880,STIEFEL_2870,86.0,F,Not Hispanic or Latino,White,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2880,2881,STIEFEL_1764,62.0,M,Not Hispanic or Latino,White,4.0,1.0,50.0,50.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2881,2882,STIEFEL_1444,61.0,M,Not Hispanic or Latino,White,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
mdasi_symptom_df["sex"] = mdasi_symptom_df["sex"].apply(lambda value: 1 if value == "M" else 0)

ic_cols = [col for col in mdasi_symptom_df.columns if "induction_c" in col.lower()]
rt_cols = [col for col in mdasi_symptom_df.columns if "rt" in col.lower()]

mdasi_symptom_df["ic"] = (~mdasi_symptom_df[ic_cols].isna().all(axis=1)).astype(int) if ic_cols else 0
mdasi_symptom_df["rt"] = (~mdasi_symptom_df[rt_cols].isna().all(axis=1)).astype(int) if rt_cols else 0

mdasi_symptom_df

,#,STIEFEL_ID,age,sex,ethnicity,race,smoking_status,tobacco_packs_per_day,tobacco_used_years,pack_years,...,4_wks_after_primar_arm_7__eq5d_complete,5_wks_after_primar_arm_7__eq5d_complete,6_months_fitbit_arm_7__eq5d_complete,day_1_chemo_arm_8__eq5d_complete,day_29_chemo_arm_8__eq5d_complete,presx_arm_8__eq5d_complete,12_month_fitbit_arm_7__eq5d_complete,1824_month_fitbit_arm_7__eq5d_complete,ic,rt
0,1,STIEFEL_2412,66.0,1,Not Hispanic or Latino,White,4.0,1.0,4.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1
1,2,STIEFEL_1728,67.0,0,Not Hispanic or Latino,White,4.0,0.5,5.0,2.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,3,STIEFEL_1729,81.0,1,Not Hispanic or Latino,White,2.0,1.0,64.4,64.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1
3,4,STIEFEL_1732,70.0,1,Not Hispanic or Latino,White,5.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1
4,5,STIEFEL_1734,64.0,0,Not Hispanic or Latino,White,4.0,0.3,4.0,1.2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2878,2879,STIEFEL_2869,64.0,1,Not Hispanic or Latino,White,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2879,2880,STIEFEL_2870,86.0,0,Not Hispanic or Latino,White,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2880,2881,STIEFEL_1764,62.0,1,Not Hispanic or Latino,White,4.0,1.0,50.0,50.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2881,2882,STIEFEL_1444,61.0,1,Not Hispanic or Latino,White,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0


In [13]:
mdasi_symptom_df = mdasi_symptom_df[mdasi_symptom_df["STIEFEL_ID"].notna()].copy()

mdasi_symptom_df["STIEFEL_ID"] = (
    mdasi_symptom_df["STIEFEL_ID"]
    .str.replace("STIEFEL_", "", regex=False)
    .astype(int)
)
mdasi_symptom_df

,#,STIEFEL_ID,age,sex,ethnicity,race,smoking_status,tobacco_packs_per_day,tobacco_used_years,pack_years,...,4_wks_after_primar_arm_7__eq5d_complete,5_wks_after_primar_arm_7__eq5d_complete,6_months_fitbit_arm_7__eq5d_complete,day_1_chemo_arm_8__eq5d_complete,day_29_chemo_arm_8__eq5d_complete,presx_arm_8__eq5d_complete,12_month_fitbit_arm_7__eq5d_complete,1824_month_fitbit_arm_7__eq5d_complete,ic,rt
0,1,2412,66.0,1,Not Hispanic or Latino,White,4.0,1.0,4.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1
1,2,1728,67.0,0,Not Hispanic or Latino,White,4.0,0.5,5.0,2.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,3,1729,81.0,1,Not Hispanic or Latino,White,2.0,1.0,64.4,64.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1
3,4,1732,70.0,1,Not Hispanic or Latino,White,5.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1
4,5,1734,64.0,0,Not Hispanic or Latino,White,4.0,0.3,4.0,1.2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2878,2879,2869,64.0,1,Not Hispanic or Latino,White,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2879,2880,2870,86.0,0,Not Hispanic or Latino,White,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2880,2881,1764,62.0,1,Not Hispanic or Latino,White,4.0,1.0,50.0,50.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2881,2882,1444,61.0,1,Not Hispanic or Latino,White,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0


In [14]:
mdasi_symptom_df.to_csv("../data/mdasi_20260421.csv", index=False)